# Problem Statement

The dataset contains detailed information about residential properties in Ames, Iowa, including physical attributes of the house, quality and condition of various components, spatial measurements, and categorical descriptions such as zoning, neighborhood, and utilities.

However, the raw data is not directly suitable for analysis in its current form. It contains missing values, inconsistent representations, and a mix of categorical and numerical variables that require proper formatting.

The objective of this step is to prepare the dataset into a clean and structured format that is suitable for further analysis. This involves handling missing values, ensuring consistency in feature representations, and organizing variables into a usable form for downstream processing.

## Loading the Data

In [ ]:
# Importing necessary libraries and setting display options
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


In [ ]:
# Loading the dataset
df_train = pd.read_csv('../data/raw/train.csv')
df_test = pd.read_csv('../data/raw/test.csv')

In [ ]:
# Basic info about the dataset
print(df_train.shape, df_test.shape)

In [ ]:
# Checking for missing values in the target variable
df_train['SalePrice'].isna().sum()

In [ ]:
# Checking if the target variable is present in the test dataset
'SalePrice' in df_test.columns

In [ ]:
# Combining train and test datasets for EDA
df = pd.concat([df_train, df_test], ignore_index=True)
df.head()

# Understanding the data

In [ ]:
# Checking the shape of the combined dataset
df.shape

In [ ]:
# Checking data types of the columns
df.dtypes

In [ ]:
# Analyzing categorical columns
for col in df.columns:
    if(df[col].dtype == 'string'):
        print(f"\n{col.upper()}")
        print(df[col].value_counts())


In [ ]:
# Analyzing numerical columns
df.describe()

## Handling the missing values in the dataset

In [ ]:
# Checking for missing values in the dataset
(df.isna().sum() / len(df) * 100).sort_values(ascending=False)

In [ ]:
# Fixing missing values in categorical columns by filling them with 'NA' as per the description of the dataset
columns_with_NA = ['Alley', 'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
for col in columns_with_NA:
    df[col] = df[col].fillna('NA')

In [ ]:
# Fixing missing values in numerical columns
df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
df.loc[df["GarageType"] == 'NA', "GarageYrBlt"] = 0
df.loc[df["MasVnrType"] == 'NA', "MasVnrArea"] = 0

In [ ]:
df[df['GarageYrBlt'].isna()][['GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond', 'GarageArea', 'GarageCars']]

# Fix special garage cases

# Row 2126: Garage clearly exists
df.loc[2126, "GarageYrBlt"] = df.loc[2126, "YearBuilt"]
df.loc[2126, "GarageFinish"] = df["GarageFinish"].mode()[0]
df.loc[2126, "GarageQual"] = "TA"
df.loc[2126, "GarageCond"] = "TA"

# Statistics for detached garages
detchd_cars_median = df.loc[
    (df["GarageType"] == "Detchd") & (df["GarageCars"].notna()),
    "GarageCars"
].median()

detchd_area_median = df.loc[
    (df["GarageType"] == "Detchd") & (df["GarageArea"].notna()),
    "GarageArea"
].median()

# Row 2576: GarageType exists but other details are missing
df.loc[2576, "GarageYrBlt"] = df.loc[2576, "YearBuilt"]
df.loc[2576, "GarageFinish"] = df["GarageFinish"].mode()[0]
df.loc[2576, "GarageQual"] = "TA"
df.loc[2576, "GarageCond"] = "TA"
df.loc[2576, "GarageCars"] = detchd_cars_median
df.loc[2576, "GarageArea"] = detchd_area_median

# Handle true "No Garage" rows

garage_cat_cols = [
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond"
]

garage_num_cols = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars"
]

for col in garage_cat_cols:
    df[col] = df[col].fillna("None")

for col in garage_num_cols:
    df[col] = df[col].fillna(0)

In [ ]:
# Handling smaller percentage of missing values in MSZoning by filling them with the mode
df["MSZoning"] = df["MSZoning"].fillna(df["MSZoning"].mode()[0])

In [ ]:
# Investigating missing values in basement-related columns
df.loc[df["BsmtHalfBath"].isna(),
       ["BsmtQual", "BsmtCond", "BsmtExposure",
        "BsmtFinType1", "BsmtFinType2"]]

df.loc[df["BsmtHalfBath"].isna(), "BsmtHalfBath"] = 0
df.loc[df["BsmtFullBath"].isna(), "BsmtFullBath"] = 0

In [ ]:
# For Functional missing values, since the percentage is very small, we can fill them with the mode which is 'Typ'
df.loc[df['Functional'].isna(), 'Functional'] = 'Typ'

In [ ]:
# For Utilities, since the percentage is very small, we can fill them with the mode which is 'AllPub'
df["Utilities"] = df["Utilities"].fillna(
    df["Utilities"].mode()[0]
)

In [ ]:
# Investigating missing values in basement area-related columns
df[df['BsmtUnfSF'].isna()][['BsmtQual']]
df['TotalBsmtSF'] = df['TotalBsmtSF'].fillna(0)
df['BsmtFinSF2'] = df['BsmtFinSF2'].fillna(0)
df['BsmtFinSF1'] = df['BsmtFinSF1'].fillna(0)
df['BsmtUnfSF'] = df['BsmtUnfSF'].fillna(0)

In [ ]:
# For the remaining categorical columns with missing values, since the percentage is very small, we can fill them with the mode
for col in ["KitchenQual", "Exterior1st", "Exterior2nd", "Electrical", "SaleType"]:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# Checking for missing values in the dataset after handling them
(df.isna().sum() / len(df) * 100).sort_values(ascending=False)

## Encoding for modelling

In [ ]:
# Creating quality maps for reasonable ordinal categorical variables
quality_map_streets = {
    "Grvl": 0,
    "Pave": 1
}

quality_map_alley = {
    "NA": 0,
    "Grvl": 1,
    "Pave": 2
}

quality_map_utilities = {
    "ELO": 0,
    "NoSeWa": 1,
    "NoSewr": 2,
    "AllPub": 3
}

quality_map_LotShape = {
    "Reg": 3,
    "IR1": 2,
    "IR2": 1,
    "IR3": 0
}

quality_map_LandSlope = {
    "Sev": 0,
    "Mod": 1,
    "Gtl": 2
}

quality_map_ExterQual = {
    "Po": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
}

quality_map_ExterCond = {
    "Po": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
}

quality_map_BsmtQual = {
    "NA": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

quality_map_BsmtCond = {
    "NA": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

quality_map_BsmtExposure = {
    "NA": 0,
    "No": 1,
    "Mn": 2,
    "Av": 3,
    "Gd": 4
}

quality_map_BsmtFinType1 = {
    "NA": 0,
    "Unf": 1,
    "LwQ": 2,
    "Rec": 3,
    "BLQ": 4,
    "ALQ": 5,
    "GLQ": 6
}

quality_map_BsmtFinType2 = quality_map_BsmtFinType1

quality_map_HeatingQC = {
    "Po": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
}

quality_map_CentralAir = {
    "N": 0,
    "Y": 1
}

quality_map_Electrical = {
    "Mix": 0,
    "FuseP": 1,
    "FuseF": 2,
    "FuseA": 3,
    "SBrkr": 4
}

quality_map_KitchenQual = {
    "Po": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
}

quality_map_Functional = {
    "Sal": 0,
    "Sev": 1,
    "Maj2": 2,
    "Maj1": 3,
    "Mod": 4,
    "Min2": 5,
    "Min1": 6,
    "Typ": 7
}

quality_map_FireplaceQu = {
    "NA": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

quality_map_GarageFinish = {
    "NA": 0,
    "Unf": 1,
    "RFn": 2,
    "Fin": 3
}

quality_map_GarageQual = {
    "NA": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

quality_map_GarageCond = quality_map_GarageQual

quality_map_PavedDrive = {
    "N": 0,
    "P": 1,
    "Y": 2
}

quality_maps_PoolQC = {
    "NA": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
}

quality_maps_Fence = {
    "NA": 0,
    "MnWw": 1,
    "GdWo": 2,
    "MnPrv": 3,
    "GdPrv": 4
}


In [ ]:
# Mapping the quality maps to the respective columns in the dataset
df['Street'] = df['Street'].map(quality_map_streets)
df['Alley'] = df['Alley'].map(quality_map_alley)
df['Utilities'] = df['Utilities'].map(quality_map_utilities)
df['LotShape'] = df['LotShape'].map(quality_map_LotShape)
df['LandSlope'] = df['LandSlope'].map(quality_map_LandSlope)
df['Functional'] = df['Functional'].map(quality_map_Functional)
df['FireplaceQu'] = df['FireplaceQu'].map(quality_map_FireplaceQu)
df['GarageFinish'] = df['GarageFinish'].map(quality_map_GarageFinish)
df['GarageQual'] = df['GarageQual'].map(quality_map_GarageQual)
df['GarageCond'] = df['GarageCond'].map(quality_map_GarageCond)
df['PavedDrive'] = df['PavedDrive'].map(quality_map_PavedDrive)
df['PoolQC'] = df['PoolQC'].map(quality_maps_PoolQC)
df['Fence'] = df['Fence'].map(quality_maps_Fence)

In [ ]:
# Ordinal encode remaining categorical features
cat_cols = df.select_dtypes(include="string").columns
df = pd.get_dummies(
    df,
    columns=cat_cols,
    drop_first=False,
    dtype=int
)

In [ ]:
# Checking data types of the columns after encoding
df.dtypes

## Feature Engineering

In [ ]:
# Making meaningful features based on domain knowledge, intuition and dataset description
df["TotalSF"] = (
    df["TotalBsmtSF"]
    + df["1stFlrSF"]
    + df["2ndFlrSF"]
)

# Total bathrooms is a more meaningful feature than individual bathroom features, and it also captures the half bathrooms as 0.5 which is more intuitive
df["TotalBath"] = (
    df["FullBath"]
    + 0.5 * df["HalfBath"]
    + df["BsmtFullBath"]
    + 0.5 * df["BsmtHalfBath"]
)

# Age of the house at the time of sale is more meaningful than the year it was built
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

# Years since remodel is more meaningful than the year it was remodeled
df["YearsSinceRemodel"] = (
    df["YrSold"] - df["YearRemodAdd"]
)

# Total porch area is more meaningful than individual porch features
df["TotalPorchSF"] = (
    df["OpenPorchSF"]
    + df["EnclosedPorch"]
    + df["3SsnPorch"]
    + df["ScreenPorch"]
)

# Outdoor area is more meaningful than individual outdoor features
df["OutdoorSF"] = (
    df["WoodDeckSF"]
    + df["OpenPorchSF"]
    + df["EnclosedPorch"]
    + df["3SsnPorch"]
    + df["ScreenPorch"]
)

# Total basement finished area is more meaningful than individual basement finished area features
df["TotalBsmtFinished"] = (
    df["BsmtFinSF1"]
    + df["BsmtFinSF2"]
)

# Total rooms above ground is more meaningful than individual room features, and it also captures the number of bedrooms which is an important feature for house prices
df["TotalRooms"] = (
    df["TotRmsAbvGrd"]
    + df["BedroomAbvGr"]
)

# Total property area is more meaningful than individual area features, and it also captures the garage area which is an important feature for house prices
df["TotalPropertySF"] = (
    df["TotalSF"]
    + df["GarageArea"]
)

# Whether the house has a garage, basement or pool is more meaningful than the area of these features, and it also captures the cases where the area is 0 but the feature exists which can be important for house prices
df["HasGarage"] = (
    df["GarageArea"] > 0
).astype(int)

# Whether the house has a basement is more meaningful than the area of the basement, and it also captures the cases where the area is 0 but the feature exists which can be important for house prices
df["HasBasement"] = (
    df["TotalBsmtSF"] > 0
).astype(int)

# Whether the house has a pool is more meaningful than the area of the pool, and it also captures the cases where the area is 0 but the feature exists which can be important for house prices
df["HasPool"] = (
    df["PoolArea"] > 0
).astype(int)

# A combined feature of quality and area can be more meaningful than individual features, as it captures the interaction between these features which can be important for house prices
df["QualitySF"] = (
    df["OverallQual"] * df["TotalSF"]
)

# Bringing the target variable to the end of the dataframe for better readability
cols = [col for col in df.columns if col != "SalePrice"] + ["SalePrice"]
df = df[cols]

In [ ]:
# ML ready dataset export
df_train = df[df['SalePrice'].notna()]
df_test = df[df['SalePrice'].isna()].drop(columns=['SalePrice'])
df_train.to_csv('../data/processed/house_price_dataset_train.csv', index=False)
df_test.to_csv('../data/processed/house_price_dataset_test.csv', index=False)
df_train.head()